![Notebook 2](images/architecture-detailed.svg)

# 2. Score real traces with AWS evaluators, in seconds

> The `Evaluate` API takes spans directly, so you can score a session from a file. **No
> agent invocation, no deploy, no CloudWatch query.** Cells here run in seconds.

📖 Story and gotchas: [`README.md`](README.md)
💻 Fixture: `fixtures/research_session_spans.json.gz`, one real session, identifiers scrubbed
⬅️ Previous: [`01_evaluate_deep_agents_locally.ipynb`](01_evaluate_deep_agents_locally.ipynb)
➡️ Next: [`03_close_the_loop_on_agentcore_runtime.ipynb`](03_close_the_loop_on_agentcore_runtime.ipynb)

---

In [ ]:
%pip install -q "bedrock-agentcore>=1.23.0" boto3

In [ ]:
import gzip, json, os, sys, time
import boto3

sys.path.insert(0, "helpers")   # the modules this notebook imports
os.environ.setdefault("AWS_REGION", "us-west-2")
REGION = os.environ["AWS_REGION"]
dp = boto3.client("bedrock-agentcore", region_name=REGION)
print("region:", REGION)

## Load one recorded session

Real spans from a real run of the notebook 1 agent. Only account and runtime identifiers
were rewritten; tool names, timings and the model's own text are untouched.

In [ ]:
SPANS = json.loads(gzip.decompress(
    open("fixtures/research_session_spans.json.gz", "rb").read()).decode())
print(f"{len(SPANS)} spans")

## The same Trajectory, now from spans

This is the pivot from notebook 1. Same object, different source — and this reader takes
no framework import, so it works for Strands or CrewAI too.

In [ ]:
from agentcore_evals import trajectory_from_spans

trajectory = trajectory_from_spans(SPANS)
print(trajectory.timeline())

## The same checks, unchanged

These are the trajectory checks from notebook 1, now scoring a production trace.

In [ ]:
from agentcore_evals import check_all, report
from research_checks import trajectory_only_checks

passed, results = check_all(trajectory_only_checks(), trajectory)
report("recorded_session", trajectory, results, passed)

## Score it with the built-in evaluators

`dp.evaluate` takes `sessionSpans` directly. Ground truth goes in as
`evaluationReferenceInputs`.

In [ ]:
def evaluate(evaluator_id, spans=SPANS, reference=None):
    """Score spans with one evaluator. Returns (value, explanation)."""
    req = {"evaluationInput": {"sessionSpans": spans}}
    if reference:
        req["evaluationReferenceInputs"] = reference
    resp = dp.evaluate(evaluatorId=evaluator_id, **req)
    results = resp.get("evaluationResults", [])
    vals = [r["value"] for r in results if isinstance(r.get("value"), (int, float))]
    expl = next((r.get("explanation") for r in results if r.get("explanation")), "")
    return (sum(vals) / len(vals) if vals else None), str(expl)

In [ ]:
for evaluator in ("Builtin.GoalSuccessRate", "Builtin.Correctness", "Builtin.Helpfulness"):
    t0 = time.time()
    value, why = evaluate(evaluator)
    print(f"{evaluator:<32} {value}   ({time.time() - t0:.1f}s)")
print(f"\nwhy (last): {why[:400]}")

## Evaluator levels decide what can be scored

`TOOL_CALL` needs tool spans. `TRACE` needs the conversation. `SESSION` looks at
everything. Ask the control plane rather than copying IDs: one AWS blog says
`Builtin.ToolSelection`, the API says `Builtin.ToolSelectionAccuracy`.

In [ ]:
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)
builtin = [e for e in ctrl.list_evaluators(maxResults=100)["evaluators"]
           if e["evaluatorType"] == "Builtin"]
for level in ("SESSION", "TRACE", "TOOL_CALL"):
    print(level, sorted(e["evaluatorId"].replace("Builtin.", "")
                        for e in builtin if e["level"] == level))

## Ground truth: assertions and an expected trajectory

`expectedTrajectory` drives the `Trajectory*Match` evaluators. `assertions` are
plain-language claims checked against the session.

In [ ]:
from research_agent import ANSWER_KEY, RULE_OF_40_RANK

SESSION_REF = [{
    "context": {"spanContext": {"sessionId": SPANS[0]["attributes"]["session.id"]}},
    "assertions": [
        {"text": "The agent researched Snowflake, Datadog and MongoDB."},
        {"text": f"It ranked them by rule of 40 as {', '.join(RULE_OF_40_RANK)}."},
        {"text": "It did not invent figures for a company it did not research."},
    ],
    # Leaf tools only, deliberately without "task". The next cell shows why.
    "expectedTrajectory": {"toolNames": ["navigate_browser", "extract_text", "execute_code"]},
}]

value, why = evaluate("Builtin.TrajectoryInOrderMatch", reference=SESSION_REF)
print(f"TrajectoryInOrderMatch (leaf tools only): {value}")
print(why[:220])

## The false negative: trajectory evaluators mis-score hierarchical agents

A parent tool span closes only when its subagent finishes, so `task` is recorded **after**
the tools it caused. Try the causally correct expectation and watch it fail.

In [ ]:
CASES = {
    "InOrder, causal (task first)":  ("Builtin.TrajectoryInOrderMatch",
                                      ["task", "navigate_browser", "extract_text", "execute_code"]),
    "InOrder, leaf tools only":      ("Builtin.TrajectoryInOrderMatch",
                                      ["navigate_browser", "extract_text", "execute_code"]),
    "AnyOrder, including task":      ("Builtin.TrajectoryAnyOrderMatch",
                                      ["task", "navigate_browser", "extract_text", "execute_code"]),
    "ExactOrder, leaf tools only":   ("Builtin.TrajectoryExactOrderMatch",
                                      ["navigate_browser", "extract_text", "execute_code"]),
}
for label, (evaluator, tools) in CASES.items():
    ref = [{**SESSION_REF[0], "expectedTrajectory": {"toolNames": tools}}]
    value, why = evaluate(evaluator, reference=ref)
    print(f"  {label:<32} {value}")

The **causally correct expectation is the one that fails**, so a 0.0 here is easy to
misread as an agent defect. `ExactOrderMatch` fails for a different reason: it demands an
exact length match, three against thirteen.

**Rule for subagent agents:** `TrajectoryInOrderMatch` with leaf tools only, or
`TrajectoryAnyOrderMatch` if you want to assert the delegation tool was used at all.

## What notebook 2 established

- The `Evaluate` API scores spans from a file, so evaluation needs no deploy
- One `Trajectory` object serves local runs and production traces
- Evaluator level decides what is scoreable
- Trajectory matchers need leaf-tools-only expectations for hierarchical agents

➡️ Next: [`03_close_the_loop_on_agentcore_runtime.ipynb`](03_close_the_loop_on_agentcore_runtime.ipynb) deploys the agent and lets
AgentCore find a bug, write a fix, and A/B it. That one costs real money and about an
hour.